<a href="https://colab.research.google.com/github/andrew-veriga/Titans_jax/blob/main/dataset_loader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset Precomputation and Loading for Titans

Этот ноутбук предназначен для фильтрации датасета TriviaQA, сохранения его в формате `ArrayRecord` и последующей быстрой загрузки для обучения модели Titans.

In [4]:
!git clone https://github.com/google-research/kauldron || true
!pip install -q ./kauldron

Cloning into 'kauldron'...
remote: Enumerating objects: 9672, done.
remote: Counting objects: 100% (183/183), done.
remote: Compressing objects: 100% (154/154), done.
remote: Total 9672 (delta 50), reused 37 (delta 29), pack-reused 9489 (from 3)
Receiving objects: 100% (9672/9672), 2.91 MiB | 14.07 MiB/s, done.
Resolving deltas: 100% (6943/6943), done.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 52.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.0/101.0 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 5.6 MB/s eta 0

In [5]:
import os
import grain.python as grain
from kauldron import kd
from etils import epath
import array_record.python.array_record_module as array_record
import tqdm
import pickle

# Константы
DATA_DIR = epath.Path('/content/drive/MyDrive/trivia_qa_filtered')
MAX_LENGTH = 1024
BATCH_SIZE = 16 # Для сохранения не критично, но используется в оригинальном коде
MAX_CONTEXT_CHARS = (MAX_LENGTH - 50) * 4

DATA_DIR.mkdir(parents=True, exist_ok=True)

## 1. Оригинальные компоненты и фильтрация

In [14]:
class FilterShortContext(grain.FilterTransform):
    def filter(self, x):
        ctxs = x.get('search_results', {}).get('search_context', [])
        if not ctxs:
            return False
        return len(ctxs[0]) <= MAX_CONTEXT_CHARS

def get_source_dataset():
    """Возвращает исходный TFDS датасет для фильтрации."""
    return kd.data.py.Tfds(
        name="trivia_qa/rc",
        split="validation", ##splits: 'test'	17,210 | 'train'	138,384 | 'validation'	18,669
        shuffle=False, # Для сохранения порядок не важен
        num_epochs=1,  # Проходим один раз
        batch_size=None, # Читаем по одной записи
    )

## 2. Сохранение в ArrayRecord

Этот этап нужно запустить один раз.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:

def precompute_and_save():
    ds = get_source_dataset()
    filter_transform = FilterShortContext()

    output_path = DATA_DIR / "validation.array_record"
    writer = array_record.ArrayRecordWriter(str(output_path), options="group_size:1")

    print(f"Начинаю фильтрацию и сохранение в {output_path}...")

    count = 0
    # Итерируемся по сырым данным из TFDS
    for record in tqdm.tqdm(ds):
        if filter_transform.filter(record):
            # Сохраняем только нужные поля для экономии места
            # Или сохраняем все, если формат format_triviaqa применится позже
            serialized = pickle.dumps(record)
            writer.write(serialized)
            count += 1

    writer.close()
    print(f"Готово! Сохранено {count} валидных записей.")

precompute_and_save() # Раскомментируйте для запуска

Начинаю фильтрацию и сохранение в /content/drive/MyDrive/trivia_qa_filtered/validation.array_record...


  0%|          | 91/18669 [00:00<00:20, 891.77it/s]

Disabling pygrain multi-processing (unsupported in colab).


100%|██████████| 18669/18669 [00:32<00:00, 571.64it/s]

Готово! Сохранено 4153 валидных записей.


## 3. Загрузка сохраненных данных

Эта функция заменяет ваш `get_train_dataset_tydi_qa` в основном цикле обучения.

In [ ]:
def get_precomputed_dataset(batch_size=16, tokenizer=None, max_length=1024, files=None):
    """Загружает и объединяет отфильтрованные данные из нескольких файлов.

    Args:
        batch_size: Размер батча.
        tokenizer: Токенизатор Gemma.
        max_length: Максимальная длина последовательности.
        files: Список имен файлов, например ['train.array_record', 'validation.array_record']
    """
    if files is None:
        files = ['train.array_record']

    paths = [str(DATA_DIR / f) for f in files]
    print(f"Загрузка данных из: {paths}")

    # 1. Определяем источник данных (DataSource)
    class PickledArrayRecordDataSource(grain.python.ArrayRecordDataSource):
        def __getitem__(self, idx):
            return pickle.loads(super().__getitem__(idx))

    data_source = PickledArrayRecordDataSource(paths)

    # 2. Создаем Kauldron Dataset
    return kd.data.py.DataSource(
        data_source=data_source,
        shuffle=True,      # Перемешивание важно при объединении разных сплитов!
        num_epochs=None,   # Бесконечно для обучения
        batch_size=batch_size,
        transforms=[
            # Здесь применяются format_triviaqa и gm.data.Seq2SeqTask
            kd.data.py.Elements(keep=["tokens", "target", "loss_mask"]),
        ],
    )

# Пример использования для объединения всех файлов:
# train_ds = get_precomputed_dataset(
#     files=['train.array_record', 'validation.array_record', 'test.array_record']
# )

## 4. Использование в kd.train.Trainer

Пример того, как это интегрируется в основной конфиг.

In [ ]:
'''
trainer = kd.train.Trainer(
    seed=42,
    workdir='./workdir',
    train_ds=get_precomputed_dataset(
        batch_size=BATCH_SIZE,
        tokenizer=tokenizer,
        max_length=MAX_LENGTH,
        files=['train.array_record', 'validation.array_record', 'test.array_record']
    ),
    model=model,
    # ... остальная конфигурация
)
'''

## 5. Офлайн-генерация ответов Gemma

В этом разделе мы пропускаем отфильтрованные записи `trivia_qa/rc` через оригинальную модель `Gemma3_1B_IT` для генерации её собственных ответов (`dst`).

In [ ]:
# ВАЖНО: Эту ячейку нужно запускать в окружении с загруженной моделью Gemma.

import array_record.python.array_record_module as array_record
import pickle
import tqdm
import kauldron as kd
import gemma.kauldron as gm
from etils import epath

# 1. Функция форматирования промпта
def format_triviaqa_prompt(x):
    ctx = ""
    if x.get('search_results', {}).get('search_context', []):
        ctx = x['search_results']['search_context'][0]
    q = x["question"]
    return f"Context: {ctx}\n\nQuestion: {q}\n\nAnswer: "

# 2. Функция генерации датасета
def generate_and_save_offline_dataset(
    model, 
    params, 
    tokenizer, 
    input_file="validation.array_record", 
    output_file="validation_gemma_generated.array_record",
    max_new_tokens=64
):
    input_path = DATA_DIR / input_file
    output_path = DATA_DIR / output_file
    
    print(f"Чтение из {input_path}")
    print(f"Сохранение в {output_path}")
    
    # Инициализация семплера Gemma
    sampler = gm.text.Sampler(
        model=model,
        params=params,
        tokenizer=tokenizer,
    )
    
    writer = array_record.ArrayRecordWriter(str(output_path), options="group_size:1")
    
    # Чтение исходного ArrayRecord
    reader = array_record.ArrayRecordReader(str(input_path))
    
    count = 0
    for record_bytes in tqdm.tqdm(reader):
        record = pickle.loads(record_bytes)
        
        # Формируем prompt
        prompt = format_triviaqa_prompt(record)
        
        # Генерируем ответ (offline)
        response_text = sampler.sample(prompt, max_new_tokens=max_new_tokens)
        
        # Подменяем эталонный ответ сгенерированным
        if 'answer' not in record:
            record['answer'] = {}
        record['answer']['value'] = response_text
        
        # Записываем обратно
        writer.write(pickle.dumps(record))
        count += 1
        
    writer.close()
    print(f"Успешно сгенерировано и сохранено {count} записей.")

# Пример запуска (раскомментировать при необходимости)
# generate_and_save_offline_dataset(
#     model=model, 
#     params=merged_params, # или оригинальные веса gemma
#     tokenizer=tokenizer,
#     input_file="validation.array_record",
#     output_file="validation_gemma_generated.array_record"
# )